# Tema 7 - Conceptos Basicos de Control PID

**Asignatura:** Fundamentos de Control - GIERM

---

## Objetivos de aprendizaje

1. Comprender el control ON-OFF y sus limitaciones (oscilacion permanente).
2. Analizar la accion **proporcional (P)** y su efecto sobre el error en regimen permanente.
3. Comprender las acciones **derivativa (D)** e **integral (I)** por separado.
4. Disenar controladores **PD**, **PI** y **PID** y comparar sus respuestas.
5. Estudiar el efecto de cada parametro ($K_p$, $T_I$, $T_D$) sobre la respuesta.
6. Aplicar los **metodos de Ziegler-Nichols** (curva de reaccion y oscilacion sostenida) para sintonizar controladores.
7. Resolver ejercicios de diseno y sintonizacion de controladores PID.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

In [ ]:
# --- Funciones auxiliares ---

def closed_loop(C_num, C_den, G_num, G_den):
    """Calcula la funcion de transferencia en lazo cerrado T(s) = C*G / (1 + C*G)."""
    # Numerador lazo abierto: C*G
    ol_num = np.polymul(C_num, G_num)
    ol_den = np.polymul(C_den, G_den)
    # Lazo cerrado: ol_num / (ol_den + ol_num)
    cl_num = ol_num
    cl_den = np.polyadd(ol_den, ol_num)
    return cl_num, cl_den

def step_response_cl(C_num, C_den, G_num, G_den, t=None):
    """Respuesta al escalon del sistema en lazo cerrado."""
    cl_num, cl_den = closed_loop(C_num, C_den, G_num, G_den)
    sys_cl = signal.TransferFunction(cl_num, cl_den)
    if t is None:
        t = np.linspace(0, 15, 2000)
    t_out, y_out = signal.step(sys_cl, T=t)
    return t_out, y_out

# Planta de referencia: G(s) = 1 / [s(s+1)] = 1 / (s^2 + s)
G_num = [1.0]
G_den = [1.0, 1.0, 0.0]  # s^2 + s

print('Funciones auxiliares cargadas.')
print(r'Planta de referencia: G(s) = 1 / [s(s+1)]')

---

## 1. Control ON-OFF (todo o nada)

El controlador mas simple posible. La senal de control solo toma dos valores:

$$\boxed{u(t) = \begin{cases} U_{\max} & \text{si } e(t) > 0 \\ U_{\min} & \text{si } e(t) < 0 \end{cases}}$$

donde $e(t) = r(t) - y(t)$ es el error.

**Caracteristicas:**
- Muy sencillo de implementar (rele, termostato).
- La salida **oscila permanentemente** alrededor de la referencia.
- No se puede ajustar la respuesta transitoria.
- Util cuando no se requiere precision (calefaccion domestica, etc.).

In [ ]:
# Simulacion de control ON-OFF sobre un sistema de primer orden con integrador
# Planta simple: dy/dt = -a*y + b*u  (primer orden)
dt = 0.01
t_sim = np.arange(0, 20, dt)
ref = 1.0
a, b = 0.5, 1.0
y_onoff = np.zeros_like(t_sim)
u_onoff = np.zeros_like(t_sim)

for i in range(1, len(t_sim)):
    e = ref - y_onoff[i-1]
    u_onoff[i] = 1.5 if e > 0 else -0.5
    dy = (-a * y_onoff[i-1] + b * u_onoff[i]) * dt
    y_onoff[i] = y_onoff[i-1] + dy

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t_sim, y_onoff, 'b-', linewidth=1.5, label=r'Salida $y(t)$')
axes[0].axhline(ref, color='r', linestyle='--', linewidth=1.2, label=r'Referencia $r=1$')
axes[0].set_ylabel(r'$y(t)$')
axes[0].set_title('Control ON-OFF: respuesta oscilante')
axes[0].legend()
axes[0].set_ylim(-0.2, 3.5)

axes[1].plot(t_sim, u_onoff, 'g-', linewidth=1.5, label=r'Senal de control $u(t)$')
axes[1].set_ylabel(r'$u(t)$')
axes[1].set_xlabel(r'Tiempo $t$ (s)')
axes[1].set_title('Senal de control ON-OFF')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 2. Controlador Proporcional (P)

La accion de control es proporcional al error:

$$\boxed{C(s) = K_p}$$

En el dominio del tiempo: $u(t) = K_p \cdot e(t)$.

**Propiedades:**
- Aumentar $K_p$ acelera la respuesta y reduce el error en regimen permanente.
- Sin embargo, **el error en regimen permanente nunca se elimina** (excepto para plantas tipo 1+ con entrada escalon).
- Un $K_p$ excesivo genera sobreoscilacion y puede desestabilizar el sistema.

Para la planta $G(s) = \frac{1}{s(s+1)}$ (tipo 1), el error ante escalon unitario con controlador P es:

$$e_{ss} = \frac{1}{1 + K_v} = 0 \quad \text{(tipo 1, escalon)}$$

Pero ante **rampa**, el error seria $e_{ss} = 1/K_p$, que no se anula.

In [ ]:
# --- PLOT 1: Efecto de Kp en controlador P ---
# Planta: G(s) = 1 / [s(s+1)]

t = np.linspace(0, 12, 2000)
Kp_values = [1, 5, 20, 100]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(12, 6))

for Kp, color in zip(Kp_values, colors):
    C_num = [Kp]
    C_den = [1.0]
    t_out, y_out = step_response_cl(C_num, C_den, G_num, G_den, t=t)
    ax.plot(t_out, y_out, color=color, linewidth=2, label=rf'$K_p = {Kp}$')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7, label='Referencia')
ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Controlador P: efecto de $K_p$ sobre la respuesta al escalon')
ax.legend(fontsize=11)
ax.set_xlim(0, 12)
ax.set_ylim(0, 1.8)

plt.tight_layout()
plt.show()

**Observaciones del grafico:**

| $K_p$ | Velocidad | Sobreoscilacion | Error $e_{ss}$ |
|-------|-----------|-----------------|----------------|
| 1     | Lenta     | Nula            | 0 (tipo 1, escalon) |
| 5     | Media     | Moderada        | 0 |
| 20    | Rapida    | Alta            | 0 |
| 100   | Muy rapida| Muy alta        | 0 |

Para esta planta tipo 1, el error ante escalon es cero para todo $K_p > 0$. Sin embargo, la **sobreoscilacion** crece dramaticamente con $K_p$. Veremos que ante **rampa**, el error si depende de $K_p$.

In [ ]:
# --- PLOT 2: Error ante rampa con controlador P (planta tipo 1) ---
# Para mostrar que P no elimina error ante rampa

t = np.linspace(0, 15, 3000)
rampa = t  # r(t) = t

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t, rampa, 'k--', linewidth=2, label='Referencia (rampa)')

for Kp, color in zip([1, 5, 20], colors[:3]):
    cl_num, cl_den = closed_loop([Kp], [1.0], G_num, G_den)
    sys_cl = signal.TransferFunction(cl_num, cl_den)
    t_out, y_out, _ = signal.lsim(sys_cl, rampa, t)
    ax.plot(t_out, y_out, color=color, linewidth=2, label=rf'$K_p = {Kp}$, $e_{{ss}} = {1/Kp:.2f}$')

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Controlador P ante entrada rampa: el error $e_{ss} = 1/K_p$ nunca se anula')
ax.legend(fontsize=11)
ax.set_xlim(0, 15)
ax.set_ylim(0, 15)

plt.tight_layout()
plt.show()

---

## 3. Controlador Derivativo (D)

$$\boxed{C(s) = K_D \cdot s}$$

En el dominio del tiempo: $u(t) = K_D \cdot \frac{de(t)}{dt}$.

**Propiedades:**
- Actua sobre la **velocidad de cambio** del error: anticipa la tendencia.
- **No puede usarse solo**: no genera accion ante error constante ($de/dt = 0$).
- Mejora la estabilidad y reduce sobreoscilacion cuando se combina con P o PI.
- Amplifica el ruido de alta frecuencia (problema practico importante).

En la practica, se utiliza siempre combinado con la accion proporcional (PD o PID).

---

## 4. Controlador Integral (I)

$$\boxed{C(s) = \frac{K_i}{s}}$$

En el dominio del tiempo: $u(t) = K_i \int_0^t e(\tau) \, d\tau$.

**Propiedades:**
- Anade un **integrador** al lazo: aumenta el tipo del sistema en 1.
- **Elimina el error en regimen permanente** ante la entrada para la cual el sistema original tenia error.
- Hace la respuesta **mas lenta y mas oscilatoria**.
- Puede causar **inestabilidad** si la ganancia integral es demasiado alta.
- El fenomeno de **windup** (saturacion del integrador) es un problema practico importante.

---

## 5. Controlador PD

Combina accion proporcional y derivativa:

$$\boxed{C(s) = K_p(1 + T_D s) = K_p + K_p T_D s}$$

donde $T_D$ es la **constante de tiempo derivativa**.

**Propiedades:**
- La accion derivativa "anticipa" el error y amortigua la respuesta.
- **Reduce la sobreoscilacion** respecto al controlador P solo.
- **No mejora el error en regimen permanente** (no anade integradores).
- Mejora la estabilidad relativa (mayor margen de fase).

In [ ]:
# --- PLOT 3: Comparacion P vs PD ---
Kp = 20
TD = 0.3
t = np.linspace(0, 10, 2000)

# Controlador P: C(s) = Kp
t_p, y_p = step_response_cl([Kp], [1.0], G_num, G_den, t=t)

# Controlador PD: C(s) = Kp*(1 + TD*s) = [Kp*TD, Kp]
C_pd_num = [Kp * TD, Kp]
C_pd_den = [1.0]
t_pd, y_pd = step_response_cl(C_pd_num, C_pd_den, G_num, G_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p, y_p, 'b-', linewidth=2.5, label=rf'P: $K_p = {Kp}$')
ax.plot(t_pd, y_pd, 'r-', linewidth=2.5, label=rf'PD: $K_p = {Kp}$, $T_D = {TD}$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7, label='Referencia')

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Comparacion P vs PD: la accion derivativa reduce la sobreoscilacion')
ax.legend(fontsize=12)
ax.set_xlim(0, 10)
ax.set_ylim(0, 1.8)

plt.tight_layout()
plt.show()

---

## 6. Controlador PI

Combina accion proporcional e integral:

$$\boxed{C(s) = K_p\left(1 + \frac{1}{T_I s}\right) = K_p \cdot \frac{T_I s + 1}{T_I s}}$$

donde $T_I$ es la **constante de tiempo integral**.

**Propiedades:**
- **Elimina el error en regimen permanente** (anade un integrador al lazo abierto).
- Hace la respuesta mas lenta y puede aumentar la sobreoscilacion.
- $T_I$ grande $\Rightarrow$ accion integral debil (lenta eliminacion del error).
- $T_I$ pequeno $\Rightarrow$ accion integral fuerte (rapida pero mas oscilatoria).

In [ ]:
# --- PLOT 4: Comparacion P vs PI ---
# Usamos planta tipo 0 para que se vea la eliminacion de error: G2(s) = 1/[(s+1)(s+2)]
G2_num = [1.0]
G2_den = [1.0, 3.0, 2.0]  # s^2 + 3s + 2

Kp = 5
TI = 2.0
t = np.linspace(0, 20, 3000)

# P: C(s) = Kp
t_p2, y_p2 = step_response_cl([Kp], [1.0], G2_num, G2_den, t=t)

# PI: C(s) = Kp*(TI*s + 1)/(TI*s) => num=[Kp*TI, Kp], den=[TI, 0]
C_pi_num = [Kp * TI, Kp]
C_pi_den = [TI, 0.0]
t_pi, y_pi = step_response_cl(C_pi_num, C_pi_den, G2_num, G2_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p2, y_p2, 'b-', linewidth=2.5, label=rf'P: $K_p = {Kp}$, $e_{{ss}} = {1/(1+Kp*0.5):.3f}$')
ax.plot(t_pi, y_pi, 'r-', linewidth=2.5, label=rf'PI: $K_p = {Kp}$, $T_I = {TI}$, $e_{{ss}} = 0$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7, label='Referencia')

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Comparacion P vs PI: la accion integral elimina el error en regimen permanente')
ax.legend(fontsize=12)
ax.set_xlim(0, 20)

plt.tight_layout()
plt.show()

In [ ]:
# --- PLOT 5: Efecto de TI en controlador PI ---
Kp = 5
TI_values = [0.5, 1.0, 3.0, 8.0]
t = np.linspace(0, 25, 3000)
colors_ti = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

fig, ax = plt.subplots(figsize=(12, 6))

for TI, color in zip(TI_values, colors_ti):
    C_pi_num = [Kp * TI, Kp]
    C_pi_den = [TI, 0.0]
    t_out, y_out = step_response_cl(C_pi_num, C_pi_den, G2_num, G2_den, t=t)
    ax.plot(t_out, y_out, color=color, linewidth=2, label=rf'$T_I = {TI}$')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7, label='Referencia')
ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(rf'Efecto de $T_I$ en controlador PI ($K_p = {Kp}$) - Planta $G(s) = 1/[(s+1)(s+2)]$')
ax.legend(fontsize=11)
ax.set_xlim(0, 25)

plt.tight_layout()
plt.show()

---

## 7. Controlador PID

Combina las tres acciones: proporcional, integral y derivativa:

$$\boxed{C(s) = K_p\left(1 + \frac{1}{T_I s} + T_D s\right) = K_p \cdot \frac{T_I T_D s^2 + T_I s + 1}{T_I s}}$$

**Propiedades:**
- La accion **P** da respuesta rapida.
- La accion **I** elimina el error en regimen permanente.
- La accion **D** reduce la sobreoscilacion y mejora la estabilidad.
- Es el controlador **mas utilizado en la industria** (>95% de los lazos de control).

La sintonizacion consiste en elegir $K_p$, $T_I$ y $T_D$ para cumplir las especificaciones.

In [ ]:
# --- PLOT 6: Gran comparacion P vs PI vs PD vs PID ---
# Planta tipo 0: G2(s) = 1/[(s+1)(s+2)] para ver efecto de I

Kp = 10
TI = 2.0
TD = 0.3
t = np.linspace(0, 15, 3000)

# P: C(s) = Kp
t_p, y_p = step_response_cl([Kp], [1.0], G2_num, G2_den, t=t)

# PI: C(s) = Kp*(TI*s + 1)/(TI*s)
C_pi_num = [Kp * TI, Kp]
C_pi_den = [TI, 0.0]
t_pi, y_pi = step_response_cl(C_pi_num, C_pi_den, G2_num, G2_den, t=t)

# PD: C(s) = Kp*(1 + TD*s) = [Kp*TD, Kp]
C_pd_num = [Kp * TD, Kp]
C_pd_den = [1.0]
t_pd, y_pd = step_response_cl(C_pd_num, C_pd_den, G2_num, G2_den, t=t)

# PID: C(s) = Kp*(TI*TD*s^2 + TI*s + 1)/(TI*s)
C_pid_num = [Kp * TI * TD, Kp * TI, Kp]
C_pid_den = [TI, 0.0]
t_pid, y_pid = step_response_cl(C_pid_num, C_pid_den, G2_num, G2_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p, y_p, 'b-', linewidth=2.5, label=r'P')
ax.plot(t_pi, y_pi, 'r-', linewidth=2.5, label=r'PI')
ax.plot(t_pd, y_pd, 'g-', linewidth=2.5, label=r'PD')
ax.plot(t_pid, y_pid, 'm-', linewidth=2.5, label=r'PID')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7, label='Referencia')

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(rf'Comparacion P vs PI vs PD vs PID ($K_p={Kp}$, $T_I={TI}$, $T_D={TD}$)')
ax.legend(fontsize=12)
ax.set_xlim(0, 15)

plt.tight_layout()
plt.show()

**Lectura del grafico:**

| Controlador | Error $e_{ss}$ | Sobreoscilacion | Velocidad |
|-------------|---------------|-----------------|----------|
| P           | $\neq 0$      | Moderada        | Rapida   |
| PI          | $= 0$         | Mayor que P     | Mas lenta |
| PD          | $\neq 0$      | Menor que P     | Similar a P |
| PID         | $= 0$         | Controlada      | Buena    |

El PID combina lo mejor de cada accion: **error cero** (I), **respuesta rapida** (P) y **baja sobreoscilacion** (D).

---

## 8. Efecto de cada parametro del PID

In [ ]:
# --- PLOT 7: Grid 2x3 - Efecto de Kp, TI, TD ---
t = np.linspace(0, 20, 3000)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Fila 1: Respuesta al escalon ---
# Columna 1: Efecto de Kp (TI=2, TD=0.3 fijos)
TI_fijo, TD_fijo = 2.0, 0.3
Kp_vals = [2, 5, 15, 40]
for Kp_v, col in zip(Kp_vals, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
    C_num = [Kp_v * TI_fijo * TD_fijo, Kp_v * TI_fijo, Kp_v]
    C_den = [TI_fijo, 0.0]
    t_o, y_o = step_response_cl(C_num, C_den, G2_num, G2_den, t=t)
    axes[0, 0].plot(t_o, y_o, color=col, linewidth=2, label=rf'$K_p={Kp_v}$')
axes[0, 0].axhline(1, color='k', linestyle='--', linewidth=0.8)
axes[0, 0].set_title(rf'Efecto de $K_p$ ($T_I={TI_fijo}$, $T_D={TD_fijo}$)')
axes[0, 0].legend(fontsize=9)
axes[0, 0].set_xlim(0, 15)
axes[0, 0].set_ylabel(r'$y(t)$')

# Columna 2: Efecto de TI (Kp=10, TD=0.3 fijos)
Kp_fijo, TD_fijo2 = 10, 0.3
TI_vals = [0.5, 1.0, 3.0, 8.0]
for TI_v, col in zip(TI_vals, ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']):
    C_num = [Kp_fijo * TI_v * TD_fijo2, Kp_fijo * TI_v, Kp_fijo]
    C_den = [TI_v, 0.0]
    t_o, y_o = step_response_cl(C_num, C_den, G2_num, G2_den, t=t)
    axes[0, 1].plot(t_o, y_o, color=col, linewidth=2, label=rf'$T_I={TI_v}$')
axes[0, 1].axhline(1, color='k', linestyle='--', linewidth=0.8)
axes[0, 1].set_title(rf'Efecto de $T_I$ ($K_p={Kp_fijo}$, $T_D={TD_fijo2}$)')
axes[0, 1].legend(fontsize=9)
axes[0, 1].set_xlim(0, 20)

# Columna 3: Efecto de TD (Kp=10, TI=2 fijos)
Kp_fijo2, TI_fijo2 = 10, 2.0
TD_vals = [0.0, 0.1, 0.3, 0.8]
for TD_v, col in zip(TD_vals, ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']):
    C_num = [Kp_fijo2 * TI_fijo2 * TD_v, Kp_fijo2 * TI_fijo2, Kp_fijo2]
    C_den = [TI_fijo2, 0.0]
    t_o, y_o = step_response_cl(C_num, C_den, G2_num, G2_den, t=t)
    axes[0, 2].plot(t_o, y_o, color=col, linewidth=2, label=rf'$T_D={TD_v}$')
axes[0, 2].axhline(1, color='k', linestyle='--', linewidth=0.8)
axes[0, 2].set_title(rf'Efecto de $T_D$ ($K_p={Kp_fijo2}$, $T_I={TI_fijo2}$)')
axes[0, 2].legend(fontsize=9)
axes[0, 2].set_xlim(0, 15)

# --- Fila 2: Tablas resumen (texto en ejes) ---
resumen = [
    (r'$K_p$ (ganancia proporcional)', 
     r'$\uparrow K_p$: respuesta mas rapida' + '\n' +
     r'$\uparrow K_p$: mayor sobreoscilacion' + '\n' +
     r'$\uparrow K_p$: puede desestabilizar' + '\n' +
     'No elimina error (en planta tipo 0)'),
    (r'$T_I$ (constante integral)',
     r'$\downarrow T_I$: accion integral fuerte' + '\n' +
     r'$\downarrow T_I$: mas oscilatorio' + '\n' +
     r'$\uparrow T_I$: accion integral debil' + '\n' +
     'Siempre elimina error en reg. permanente'),
    (r'$T_D$ (constante derivativa)',
     r'$\uparrow T_D$: mayor amortiguamiento' + '\n' +
     r'$\uparrow T_D$: reduce sobreoscilacion' + '\n' +
     r'$\uparrow T_D$ excesivo: respuesta lenta' + '\n' +
     'Sensible al ruido de alta frecuencia')
]

for j, (titulo, texto) in enumerate(resumen):
    axes[1, j].axis('off')
    axes[1, j].text(0.5, 0.75, titulo, transform=axes[1, j].transAxes,
                    ha='center', va='top', fontsize=13, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    axes[1, j].text(0.5, 0.45, texto, transform=axes[1, j].transAxes,
                    ha='center', va='top', fontsize=11, family='monospace')

fig.suptitle('Efecto de cada parametro del PID sobre la respuesta al escalon', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 9. Metodo de Ziegler-Nichols 1: Curva de reaccion

Aplicable a plantas con respuesta al escalon en forma de **S** (plantas estables sin integradores).

### Procedimiento

1. Se aplica un **escalon** a la planta en lazo abierto.
2. Se traza la **tangente** en el punto de maxima pendiente.
3. Se determinan tres parametros de la curva:
   - $K$: ganancia estatica (valor final de la respuesta / amplitud del escalon).
   - $L$: **retardo** (interseccion de la tangente con el eje de tiempo).
   - $T$: **constante de tiempo** (desde $L$ hasta donde la tangente alcanza el valor final).

### Tabla de sintonizacion de Ziegler-Nichols (Metodo 1)

| Controlador | $K_p$ | $T_I$ | $T_D$ |
|-------------|-------|-------|-------|
| P           | $\dfrac{T}{KL}$ | $-$ | $-$ |
| PI          | $0.9\dfrac{T}{KL}$ | $\dfrac{L}{0.3}$ | $-$ |
| PID         | $1.2\dfrac{T}{KL}$ | $2L$ | $0.5L$ |

In [ ]:
# --- PLOT 8: Curva de reaccion con K, L, T marcados ---
# Planta ejemplo: G(s) = 2 / [(s+1)(0.5s+1)] = 2 / (0.5s^2 + 1.5s + 1)
G_zn_num = [2.0]
G_zn_den = [0.5, 1.5, 1.0]

sys_zn = signal.TransferFunction(G_zn_num, G_zn_den)
t_zn = np.linspace(0, 8, 2000)
t_out, y_out = signal.step(sys_zn, T=t_zn)

# Ganancia estatica
K = y_out[-1]

# Punto de maxima pendiente
dy = np.gradient(y_out, t_out)
idx_max = np.argmax(dy)
t_inflection = t_out[idx_max]
y_inflection = y_out[idx_max]
slope = dy[idx_max]

# Tangente: y = slope*(t - t_inflection) + y_inflection
t_tang = np.linspace(0, 8, 500)
y_tang = slope * (t_tang - t_inflection) + y_inflection

# L: interseccion de tangente con y=0
L = t_inflection - y_inflection / slope
# T: interseccion de tangente con y=K
T_param = (K - y_inflection) / slope + t_inflection
T_val = T_param - L

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_out, y_out, 'b-', linewidth=2.5, label='Respuesta al escalon (curva de reaccion)')
ax.axhline(K, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)

# Tangente (solo parte visible)
mask = (y_tang >= -0.1) & (y_tang <= K + 0.3)
ax.plot(t_tang[mask], y_tang[mask], 'r--', linewidth=2, label='Tangente en punto de inflexion')

# Marcar L
ax.annotate('', xy=(L, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(L/2, -0.15, rf'$L = {L:.2f}$', ha='center', fontsize=12, color='green', fontweight='bold')

# Marcar T
ax.annotate('', xy=(T_param, K), xytext=(L, 0),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text((L + T_param)/2 + 0.2, K/2, rf'$T = {T_val:.2f}$', fontsize=12, color='purple', fontweight='bold')

# Marcar K
ax.annotate(rf'$K = {K:.2f}$', xy=(7, K), fontsize=12, color='gray', fontweight='bold',
            va='bottom')

# Punto de inflexion
ax.plot(t_inflection, y_inflection, 'ro', markersize=8, zorder=5)

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title('Metodo de Ziegler-Nichols 1: Curva de reaccion')
ax.legend(fontsize=11)
ax.set_xlim(-0.2, 8)
ax.set_ylim(-0.3, K + 0.5)

plt.tight_layout()
plt.show()

print(f'Parametros de la curva de reaccion:')
print(f'  K = {K:.4f}')
print(f'  L = {L:.4f} s')
print(f'  T = {T_val:.4f} s')

### Ejemplo resuelto: Ziegler-Nichols Metodo 1

**Enunciado:** Para la planta $G(s) = \dfrac{2}{(s+1)(0.5s+1)}$, aplicar el metodo de la curva de reaccion para disenar controladores P, PI y PID.

In [ ]:
# Ejemplo resuelto Ziegler-Nichols Metodo 1
print('=== Ziegler-Nichols Metodo 1: Curva de reaccion ===')
print(f'Planta: G(s) = 2 / [(s+1)(0.5s+1)]')
print(f'Parametros: K = {K:.4f}, L = {L:.4f}, T = {T_val:.4f}')
print()

# P
Kp_P = T_val / (K * L)
print(f'--- Controlador P ---')
print(f'  Kp = T/(K*L) = {T_val:.4f}/({K:.4f}*{L:.4f}) = {Kp_P:.4f}')
print()

# PI
Kp_PI = 0.9 * T_val / (K * L)
TI_PI = L / 0.3
print(f'--- Controlador PI ---')
print(f'  Kp = 0.9*T/(K*L) = {Kp_PI:.4f}')
print(f'  TI = L/0.3 = {L:.4f}/0.3 = {TI_PI:.4f}')
print()

# PID
Kp_PID = 1.2 * T_val / (K * L)
TI_PID = 2 * L
TD_PID = 0.5 * L
print(f'--- Controlador PID ---')
print(f'  Kp = 1.2*T/(K*L) = {Kp_PID:.4f}')
print(f'  TI = 2*L = {TI_PID:.4f}')
print(f'  TD = 0.5*L = {TD_PID:.4f}')

In [ ]:
# Comparacion de los controladores ZN Metodo 1
t = np.linspace(0, 15, 3000)

# P
t_p, y_p = step_response_cl([Kp_P], [1.0], G_zn_num, G_zn_den, t=t)

# PI
C_pi_n = [Kp_PI * TI_PI, Kp_PI]
C_pi_d = [TI_PI, 0.0]
t_pi, y_pi = step_response_cl(C_pi_n, C_pi_d, G_zn_num, G_zn_den, t=t)

# PID
C_pid_n = [Kp_PID * TI_PID * TD_PID, Kp_PID * TI_PID, Kp_PID]
C_pid_d = [TI_PID, 0.0]
t_pid, y_pid = step_response_cl(C_pid_n, C_pid_d, G_zn_num, G_zn_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p, y_p, 'b-', linewidth=2, label=rf'P: $K_p={Kp_P:.2f}$')
ax.plot(t_pi, y_pi, 'r-', linewidth=2, label=rf'PI: $K_p={Kp_PI:.2f}$, $T_I={TI_PI:.2f}$')
ax.plot(t_pid, y_pid, 'g-', linewidth=2, label=rf'PID: $K_p={Kp_PID:.2f}$, $T_I={TI_PID:.2f}$, $T_D={TD_PID:.2f}$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title('Ziegler-Nichols Metodo 1: Comparacion P vs PI vs PID')
ax.legend(fontsize=11)
ax.set_xlim(0, 15)

plt.tight_layout()
plt.show()

---

## 10. Metodo de Ziegler-Nichols 2: Oscilacion sostenida

Aplicable a plantas donde se puede aumentar la ganancia hasta alcanzar el **limite de estabilidad**.

### Procedimiento

1. Con solo control proporcional ($C(s) = K_p$), aumentar $K_p$ hasta que el sistema en lazo cerrado **oscile de forma sostenida** (estabilidad marginal).
2. La ganancia critica se denomina $K_{cr}$ y el periodo de oscilacion $T_{cr}$.
3. Usar la tabla de sintonizacion.

### Tabla de sintonizacion de Ziegler-Nichols (Metodo 2)

| Controlador | $K_p$ | $T_I$ | $T_D$ |
|-------------|-------|-------|-------|
| P           | $0.5 K_{cr}$ | $-$ | $-$ |
| PI          | $0.45 K_{cr}$ | $\dfrac{T_{cr}}{1.2}$ | $-$ |
| PID         | $0.6 K_{cr}$ | $\dfrac{T_{cr}}{2}$ | $\dfrac{T_{cr}}{8}$ |

In [ ]:
# --- PLOT 9: Oscilacion sostenida ---
# Planta: G(s) = 1 / [s(s+1)(0.2s+1)] = 1 / (0.2s^3 + 1.2s^2 + s)
G_zn2_num = [1.0]
G_zn2_den = [0.2, 1.2, 1.0, 0.0]  # 0.2s^3 + 1.2s^2 + s

# Para encontrar Kcr, usamos Routh-Hurwitz o busqueda numerica
# Ecuacion caracteristica con C=Kp: 0.2s^3 + 1.2s^2 + s + Kp = 0
# Routh: fila s^1: (1.2*1 - 0.2*Kp)/1.2 > 0 => Kp < 6
# Kcr = 6
Kcr = 6.0

# Frecuencia de oscilacion: sustituir s=jw en ecuacion caracteristica
# 0.2(jw)^3 + 1.2(jw)^2 + jw + 6 = 0
# Parte real: -1.2w^2 + 6 = 0 => w = sqrt(5) rad/s
# Tcr = 2*pi/w
w_cr = np.sqrt(5)
Tcr = 2 * np.pi / w_cr

# Simular oscilacion sostenida
t_zn2 = np.linspace(0, 20, 5000)
cl_num_cr, cl_den_cr = closed_loop([Kcr], [1.0], G_zn2_num, G_zn2_den)
sys_cr = signal.TransferFunction(cl_num_cr, cl_den_cr)
t_cr, y_cr = signal.step(sys_cr, T=t_zn2)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_cr, y_cr, 'b-', linewidth=2, label=rf'Oscilacion sostenida con $K_{{cr}} = {Kcr}$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.5)

# Marcar Tcr
# Encontrar picos para marcar periodo
from scipy.signal import find_peaks
peaks, _ = find_peaks(y_cr, height=1.0)
if len(peaks) >= 2:
    t1, t2 = t_cr[peaks[0]], t_cr[peaks[1]]
    y_peak = y_cr[peaks[0]]
    ax.annotate('', xy=(t2, y_peak + 0.05), xytext=(t1, y_peak + 0.05),
                arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax.text((t1 + t2)/2, y_peak + 0.12, rf'$T_{{cr}} = {Tcr:.2f}$ s',
            ha='center', fontsize=12, color='red', fontweight='bold')

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(rf'Ziegler-Nichols Metodo 2: Oscilacion sostenida ($K_{{cr}}={Kcr}$, $T_{{cr}}={Tcr:.2f}$ s)')
ax.legend(fontsize=11)
ax.set_xlim(0, 20)

plt.tight_layout()
plt.show()

print(f'Parametros criticos:')
print(f'  Kcr = {Kcr}')
print(f'  w_cr = sqrt(5) = {w_cr:.4f} rad/s')
print(f'  Tcr = 2*pi/w_cr = {Tcr:.4f} s')

### Ejemplo resuelto: Ziegler-Nichols Metodo 2

**Enunciado:** Para la planta $G(s) = \dfrac{1}{s(s+1)(0.2s+1)}$, aplicar el metodo de la oscilacion sostenida para disenar controladores P, PI y PID.

In [ ]:
# Ejemplo resuelto Ziegler-Nichols Metodo 2
print('=== Ziegler-Nichols Metodo 2: Oscilacion sostenida ===')
print(f'Planta: G(s) = 1 / [s(s+1)(0.2s+1)]')
print(f'Parametros criticos: Kcr = {Kcr}, Tcr = {Tcr:.4f} s')
print()

# P
Kp_P2 = 0.5 * Kcr
print(f'--- Controlador P ---')
print(f'  Kp = 0.5*Kcr = 0.5*{Kcr} = {Kp_P2:.4f}')
print()

# PI
Kp_PI2 = 0.45 * Kcr
TI_PI2 = Tcr / 1.2
print(f'--- Controlador PI ---')
print(f'  Kp = 0.45*Kcr = {Kp_PI2:.4f}')
print(f'  TI = Tcr/1.2 = {Tcr:.4f}/1.2 = {TI_PI2:.4f}')
print()

# PID
Kp_PID2 = 0.6 * Kcr
TI_PID2 = Tcr / 2
TD_PID2 = Tcr / 8
print(f'--- Controlador PID ---')
print(f'  Kp = 0.6*Kcr = {Kp_PID2:.4f}')
print(f'  TI = Tcr/2 = {TI_PID2:.4f}')
print(f'  TD = Tcr/8 = {TD_PID2:.4f}')

In [ ]:
# --- PLOT 10: Comparacion ZN Metodo 2 ---
t = np.linspace(0, 20, 3000)

# P
t_p2, y_p2 = step_response_cl([Kp_P2], [1.0], G_zn2_num, G_zn2_den, t=t)

# PI
C_pi2_n = [Kp_PI2 * TI_PI2, Kp_PI2]
C_pi2_d = [TI_PI2, 0.0]
t_pi2, y_pi2 = step_response_cl(C_pi2_n, C_pi2_d, G_zn2_num, G_zn2_den, t=t)

# PID
C_pid2_n = [Kp_PID2 * TI_PID2 * TD_PID2, Kp_PID2 * TI_PID2, Kp_PID2]
C_pid2_d = [TI_PID2, 0.0]
t_pid2, y_pid2 = step_response_cl(C_pid2_n, C_pid2_d, G_zn2_num, G_zn2_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p2, y_p2, 'b-', linewidth=2, label=rf'P: $K_p={Kp_P2:.2f}$')
ax.plot(t_pi2, y_pi2, 'r-', linewidth=2, label=rf'PI: $K_p={Kp_PI2:.2f}$, $T_I={TI_PI2:.2f}$')
ax.plot(t_pid2, y_pid2, 'g-', linewidth=2, label=rf'PID: $K_p={Kp_PID2:.2f}$, $T_I={TI_PID2:.2f}$, $T_D={TD_PID2:.2f}$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title('Ziegler-Nichols Metodo 2: Comparacion P vs PI vs PID')
ax.legend(fontsize=11)
ax.set_xlim(0, 20)

plt.tight_layout()
plt.show()

---

## 11. Ejercicios resueltos

### Ejercicio 1: Diseno de controlador P para especificacion de error

**Enunciado:** Para la planta $G(s) = \dfrac{4}{(s+2)(s+4)}$, disenar un controlador proporcional $C(s) = K_p$ tal que el error ante rampa sea menor que $0.1$.

**Solucion:**

La planta es tipo 0. El error ante rampa con controlador P es $e_{ss} = \infty$ (no se puede). Sin embargo, si la referencia es un **escalon**, el error es:

$$e_{ss} = \frac{1}{1 + K_p \cdot G(0)} = \frac{1}{1 + K_p \cdot \frac{4}{2 \cdot 4}} = \frac{1}{1 + 0.5 K_p}$$

Para $e_{ss} < 0.1$:

$$\frac{1}{1 + 0.5 K_p} < 0.1 \implies 1 + 0.5 K_p > 10 \implies K_p > 18$$

$$\boxed{K_p > 18}$$

In [ ]:
# Ejercicio 1: Verificacion
G_e1_num = [4.0]
G_e1_den = [1.0, 6.0, 8.0]  # (s+2)(s+4) = s^2 + 6s + 8
t = np.linspace(0, 10, 2000)

fig, ax = plt.subplots(figsize=(12, 6))

for Kp_v, col in zip([5, 10, 18, 30], ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
    t_o, y_o = step_response_cl([Kp_v], [1.0], G_e1_num, G_e1_den, t=t)
    e_ss = 1 / (1 + Kp_v * 0.5)
    ax.plot(t_o, y_o, color=col, linewidth=2, label=rf'$K_p={Kp_v}$, $e_{{ss}}={e_ss:.3f}$')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(0.9, color='gray', linestyle=':', linewidth=1, alpha=0.7, label=r'Limite $e_{ss} = 0.1$')
ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Ejercicio 1: Diseno de P para $e_{ss} < 0.1$ ante escalon')
ax.legend(fontsize=10)
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

### Ejercicio 2: Diseno de controlador PI

**Enunciado:** Para la planta $G(s) = \dfrac{4}{(s+2)(s+4)}$, disenar un controlador PI que elimine el error en regimen permanente ante escalon y tenga una respuesta aceptable.

**Solucion:**

Con controlador PI: $C(s) = K_p\left(1 + \dfrac{1}{T_I s}\right) = K_p \cdot \dfrac{T_I s + 1}{T_I s}$

El lazo abierto tiene un integrador (del PI), por lo que el sistema es tipo 1 y $e_{ss} = 0$ ante escalon para todo $K_p > 0$ (siempre que sea estable).

Elegimos $K_p = 5$, $T_I = 1.5$ (compromiso entre velocidad y sobreoscilacion):

$$\boxed{C(s) = 5\left(1 + \frac{1}{1.5s}\right) = 5 \cdot \frac{1.5s + 1}{1.5s}}$$

In [ ]:
# Ejercicio 2: Verificacion PI
Kp_e2 = 5
TI_e2 = 1.5
t = np.linspace(0, 10, 2000)

# P solo
t_p, y_p = step_response_cl([Kp_e2], [1.0], G_e1_num, G_e1_den, t=t)

# PI
C_pi_n = [Kp_e2 * TI_e2, Kp_e2]
C_pi_d = [TI_e2, 0.0]
t_pi, y_pi = step_response_cl(C_pi_n, C_pi_d, G_e1_num, G_e1_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_p, y_p, 'b-', linewidth=2, label=rf'P: $K_p={Kp_e2}$, $e_{{ss}} \neq 0$')
ax.plot(t_pi, y_pi, 'r-', linewidth=2, label=rf'PI: $K_p={Kp_e2}$, $T_I={TI_e2}$, $e_{{ss}} = 0$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title('Ejercicio 2: Controlador PI elimina error en regimen permanente')
ax.legend(fontsize=11)
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

### Ejercicio 3: Diseno de controlador PID

**Enunciado:** Para la planta $G(s) = \dfrac{4}{(s+2)(s+4)}$, disenar un controlador PID que elimine el error y tenga baja sobreoscilacion.

**Solucion:**

Partimos del PI del ejercicio anterior y anadimos accion derivativa para reducir la sobreoscilacion.

$$C(s) = K_p\left(1 + \frac{1}{T_I s} + T_D s\right)$$

Con $K_p = 5$, $T_I = 1.5$, $T_D = 0.15$:

$$\boxed{C(s) = 5\left(1 + \frac{1}{1.5s} + 0.15s\right)}$$

In [ ]:
# Ejercicio 3: PID
Kp_e3 = 5
TI_e3 = 1.5
TD_e3 = 0.15
t = np.linspace(0, 10, 2000)

# PI (ejercicio anterior)
C_pi_n = [Kp_e3 * TI_e3, Kp_e3]
C_pi_d = [TI_e3, 0.0]
t_pi, y_pi = step_response_cl(C_pi_n, C_pi_d, G_e1_num, G_e1_den, t=t)

# PID
C_pid_n = [Kp_e3 * TI_e3 * TD_e3, Kp_e3 * TI_e3, Kp_e3]
C_pid_d = [TI_e3, 0.0]
t_pid, y_pid = step_response_cl(C_pid_n, C_pid_d, G_e1_num, G_e1_den, t=t)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_pi, y_pi, 'r-', linewidth=2, label=rf'PI: $K_p={Kp_e3}$, $T_I={TI_e3}$')
ax.plot(t_pid, y_pid, 'g-', linewidth=2, label=rf'PID: $K_p={Kp_e3}$, $T_I={TI_e3}$, $T_D={TD_e3}$')
ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)

ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title('Ejercicio 3: PID reduce sobreoscilacion respecto a PI')
ax.legend(fontsize=11)
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

### Ejercicio 4: Ziegler-Nichols Metodo 1 completo

**Enunciado:** La respuesta al escalon de una planta da una curva en S con los siguientes parametros: $K = 1.5$, $L = 0.3$ s, $T = 1.8$ s. Disenar controladores P, PI y PID por Ziegler-Nichols.

**Solucion:**

In [ ]:
# Ejercicio 4: ZN Metodo 1 con datos
K_e4 = 1.5
L_e4 = 0.3
T_e4 = 1.8

print('=== Ejercicio 4: Ziegler-Nichols Metodo 1 ===')
print(f'Datos: K = {K_e4}, L = {L_e4} s, T = {T_e4} s')
print()

# P
Kp_e4_P = T_e4 / (K_e4 * L_e4)
print(f'--- Controlador P ---')
print(f'  Kp = T/(K*L) = {T_e4}/({K_e4}*{L_e4}) = {Kp_e4_P:.4f}')
print()

# PI
Kp_e4_PI = 0.9 * T_e4 / (K_e4 * L_e4)
TI_e4_PI = L_e4 / 0.3
print(f'--- Controlador PI ---')
print(f'  Kp = 0.9*T/(K*L) = {Kp_e4_PI:.4f}')
print(f'  TI = L/0.3 = {L_e4}/0.3 = {TI_e4_PI:.4f}')
print(f'  C(s) = {Kp_e4_PI:.2f} * (1 + 1/({TI_e4_PI:.2f}*s))')
print()

# PID
Kp_e4_PID = 1.2 * T_e4 / (K_e4 * L_e4)
TI_e4_PID = 2 * L_e4
TD_e4_PID = 0.5 * L_e4
print(f'--- Controlador PID ---')
print(f'  Kp = 1.2*T/(K*L) = {Kp_e4_PID:.4f}')
print(f'  TI = 2*L = {TI_e4_PID:.4f}')
print(f'  TD = 0.5*L = {TD_e4_PID:.4f}')
print(f'  C(s) = {Kp_e4_PID:.2f} * (1 + 1/({TI_e4_PID:.2f}*s) + {TD_e4_PID:.2f}*s)')

### Ejercicio 5: Ziegler-Nichols Metodo 2 completo

**Enunciado:** Al aumentar la ganancia proporcional sobre una planta, se alcanza oscilacion sostenida con $K_{cr} = 10$ y periodo $T_{cr} = 1.5$ s. Disenar controladores P, PI y PID.

**Solucion:**

In [ ]:
# Ejercicio 5: ZN Metodo 2 con datos
Kcr_e5 = 10
Tcr_e5 = 1.5

print('=== Ejercicio 5: Ziegler-Nichols Metodo 2 ===')
print(f'Datos: Kcr = {Kcr_e5}, Tcr = {Tcr_e5} s')
print()

# P
Kp_e5_P = 0.5 * Kcr_e5
print(f'--- Controlador P ---')
print(f'  Kp = 0.5*Kcr = 0.5*{Kcr_e5} = {Kp_e5_P:.4f}')
print()

# PI
Kp_e5_PI = 0.45 * Kcr_e5
TI_e5_PI = Tcr_e5 / 1.2
print(f'--- Controlador PI ---')
print(f'  Kp = 0.45*Kcr = {Kp_e5_PI:.4f}')
print(f'  TI = Tcr/1.2 = {Tcr_e5}/1.2 = {TI_e5_PI:.4f}')
print(f'  C(s) = {Kp_e5_PI:.2f} * (1 + 1/({TI_e5_PI:.2f}*s))')
print()

# PID
Kp_e5_PID = 0.6 * Kcr_e5
TI_e5_PID = Tcr_e5 / 2
TD_e5_PID = Tcr_e5 / 8
print(f'--- Controlador PID ---')
print(f'  Kp = 0.6*Kcr = {Kp_e5_PID:.4f}')
print(f'  TI = Tcr/2 = {TI_e5_PID:.4f}')
print(f'  TD = Tcr/8 = {TD_e5_PID:.4f}')
print(f'  C(s) = {Kp_e5_PID:.2f} * (1 + 1/({TI_e5_PID:.2f}*s) + {TD_e5_PID:.4f}*s)')

### Ejercicio 6: Comparacion de controladores

**Enunciado:** Para la planta $G(s) = \dfrac{1}{(s+1)(s+3)}$, comparar las respuestas al escalon de los controladores P ($K_p=10$), PI ($K_p=10$, $T_I=2$), PD ($K_p=10$, $T_D=0.2$) y PID ($K_p=10$, $T_I=2$, $T_D=0.2$). Medir el error en regimen permanente y la sobreoscilacion.

In [ ]:
# Ejercicio 6: Comparacion completa
G_e6_num = [1.0]
G_e6_den = [1.0, 4.0, 3.0]  # (s+1)(s+3) = s^2 + 4s + 3
Kp_e6 = 10
TI_e6 = 2.0
TD_e6 = 0.2
t = np.linspace(0, 12, 3000)

controllers = {
    'P':   ([Kp_e6], [1.0]),
    'PI':  ([Kp_e6 * TI_e6, Kp_e6], [TI_e6, 0.0]),
    'PD':  ([Kp_e6 * TD_e6, Kp_e6], [1.0]),
    'PID': ([Kp_e6 * TI_e6 * TD_e6, Kp_e6 * TI_e6, Kp_e6], [TI_e6, 0.0]),
}
colores = {'P': '#1f77b4', 'PI': '#d62728', 'PD': '#2ca02c', 'PID': '#9467bd'}

fig, ax = plt.subplots(figsize=(12, 6))

for nombre, (c_num, c_den) in controllers.items():
    t_o, y_o = step_response_cl(c_num, c_den, G_e6_num, G_e6_den, t=t)
    e_ss = abs(1.0 - y_o[-1])
    overshoot = (np.max(y_o) - y_o[-1]) / y_o[-1] * 100 if y_o[-1] > 0.01 else 0
    ax.plot(t_o, y_o, color=colores[nombre], linewidth=2.5,
            label=rf'{nombre}: $e_{{ss}}={e_ss:.3f}$, OS$={overshoot:.1f}\%$')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Ejercicio 6: Comparacion P vs PI vs PD vs PID - $G(s)=1/[(s+1)(s+3)]$')
ax.legend(fontsize=10)
ax.set_xlim(0, 12)

plt.tight_layout()
plt.show()

---

## 12. Catalogo de problemas tipo

A continuacion se presenta un catalogo de los tipos de problemas mas frecuentes en control PID, cada uno con su metodologia paso a paso y un ejemplo resuelto.

### Tipo 1: Calcular el error en regimen permanente con controlador P

**Metodologia:**
1. Identificar el tipo de la planta $G(s)$ (numero de integradores).
2. Calcular la constante de error relevante ($K_p$, $K_v$, $K_a$) del lazo abierto $C(s)G(s)$.
3. Aplicar la formula del error: $e_{ss} = \dfrac{1}{1+K_p}$ (escalon, tipo 0), $e_{ss} = \dfrac{1}{K_v}$ (rampa, tipo 1), etc.

**Ejemplo:** $G(s) = \dfrac{5}{(s+1)(s+5)}$, $C(s) = 8$. Error ante escalon.

$K_p = \lim_{s \to 0} C(s)G(s) = 8 \cdot \dfrac{5}{1 \cdot 5} = 8$

$$\boxed{e_{ss} = \frac{1}{1+8} = \frac{1}{9} \approx 0.111}$$

### Tipo 2: Disenar Kp para especificacion de error

**Metodologia:**
1. Expresar $e_{ss}$ en funcion de $K_p$.
2. Despejar $K_p$ de la desigualdad $e_{ss} < e_{\max}$.

**Ejemplo:** $G(s) = \dfrac{3}{(s+2)(s+6)}$. Disenar $K_p$ para $e_{ss} < 0.05$ ante escalon.

$G(0) = \dfrac{3}{12} = 0.25$. Luego $e_{ss} = \dfrac{1}{1 + 0.25 K_p} < 0.05$

$1 + 0.25 K_p > 20 \implies K_p > 76$

$$\boxed{K_p > 76}$$

### Tipo 3: Disenar controlador PI para error cero

**Metodologia:**
1. El PI anade un integrador $\Rightarrow$ aumenta el tipo del sistema en 1.
2. Elegir $K_p$ y $T_I$ para estabilidad y prestaciones aceptables.
3. Verificar estabilidad (Routh-Hurwitz o simulacion).

**Ejemplo:** $G(s) = \dfrac{2}{(s+1)(s+4)}$. Disenar PI para $e_{ss} = 0$ ante escalon.

$C(s) = K_p \cdot \dfrac{T_I s + 1}{T_I s}$. Con $K_p = 4$, $T_I = 1$:

$$\boxed{C(s) = 4 \cdot \frac{s + 1}{s} = \frac{4s + 4}{s}}$$

El cero del PI en $s = -1$ cancela el polo de la planta, simplificando el diseno.

### Tipo 4: Disenar controlador PD para reducir sobreoscilacion

**Metodologia:**
1. Disenar primero un P que cumpla la velocidad de respuesta.
2. Anadir accion D ($T_D$) para reducir sobreoscilacion.
3. Iterar $T_D$ hasta cumplir la especificacion.

**Ejemplo:** $G(s) = \dfrac{1}{s(s+2)}$. Con $K_p = 20$, la sobreoscilacion es excesiva. Disenar PD para $M_p < 10\%$.

Lazo cerrado con PD: $T(s) = \dfrac{K_p(1+T_D s)}{s^2 + (2 + K_p T_D)s + K_p}$

Para $M_p < 10\%$: $\zeta > 0.59$. Como $\omega_n = \sqrt{20}$, $2\zeta\omega_n = 2 + 20 T_D$.

$T_D = \dfrac{2 \cdot 0.59 \cdot \sqrt{20} - 2}{20} = \dfrac{5.28 - 2}{20} = 0.164$

$$\boxed{T_D \geq 0.164 \text{ s}}$$

### Tipo 5: Disenar PID completo

**Metodologia:**
1. Elegir $K_p$ para velocidad de respuesta.
2. Anadir accion I ($T_I$) para eliminar error.
3. Anadir accion D ($T_D$) para controlar sobreoscilacion.
4. Ajustar iterativamente.

**Ejemplo:** $G(s) = \dfrac{1}{(s+1)(s+3)}$. Disenar PID con $e_{ss} = 0$, $M_p < 20\%$.

Con $K_p = 15$, $T_I = 1$, $T_D = 0.15$:

$$\boxed{C(s) = 15\left(1 + \frac{1}{s} + 0.15s\right) = 15 \cdot \frac{0.15s^2 + s + 1}{s}}$$

In [ ]:
# Verificacion Tipo 5
G_t5_num = [1.0]
G_t5_den = [1.0, 4.0, 3.0]
Kp_t5, TI_t5, TD_t5 = 15, 1.0, 0.15
t = np.linspace(0, 8, 2000)

C_pid_n = [Kp_t5 * TI_t5 * TD_t5, Kp_t5 * TI_t5, Kp_t5]
C_pid_d = [TI_t5, 0.0]
t_o, y_o = step_response_cl(C_pid_n, C_pid_d, G_t5_num, G_t5_den, t=t)

e_ss = abs(1 - y_o[-1])
Mp = (np.max(y_o) - 1.0) * 100
print(f'PID: Kp={Kp_t5}, TI={TI_t5}, TD={TD_t5}')
print(f'  Error regimen permanente: {e_ss:.6f}')
print(f'  Sobreoscilacion: {Mp:.1f}%')
print(f'  Cumple especificaciones: e_ss=0 ({"SI" if e_ss < 0.01 else "NO"}), Mp<20% ({"SI" if Mp < 20 else "NO"})')

### Tipo 6: Sintonizacion por Ziegler-Nichols Metodo 1 (datos de la curva)

**Metodologia:**
1. Obtener $K$, $L$, $T$ de la curva de reaccion.
2. Aplicar la tabla de ZN Metodo 1.
3. Verificar la respuesta.

**Ejemplo:** Curva de reaccion con $K=2$, $L=0.5$ s, $T=3$ s.

PID: $K_p = 1.2 \cdot \dfrac{3}{2 \cdot 0.5} = 3.6$, $T_I = 2 \cdot 0.5 = 1.0$, $T_D = 0.5 \cdot 0.5 = 0.25$

$$\boxed{C(s) = 3.6\left(1 + \frac{1}{s} + 0.25s\right)}$$

### Tipo 7: Sintonizacion por Ziegler-Nichols Metodo 2 (datos criticos)

**Metodologia:**
1. Obtener $K_{cr}$ y $T_{cr}$ experimentalmente o por Routh-Hurwitz.
2. Aplicar la tabla de ZN Metodo 2.
3. Verificar la respuesta.

**Ejemplo:** $K_{cr} = 15$, $T_{cr} = 2$ s.

PID: $K_p = 0.6 \cdot 15 = 9$, $T_I = 2/2 = 1$, $T_D = 2/8 = 0.25$

$$\boxed{C(s) = 9\left(1 + \frac{1}{s} + 0.25s\right)}$$

### Tipo 8: Obtener Kcr y Tcr por Routh-Hurwitz

**Metodologia:**
1. Escribir la ecuacion caracteristica con $C(s) = K_p$: $1 + K_p G(s) = 0$.
2. Aplicar criterio de Routh para encontrar $K_{cr}$ (valor que hace una fila cero).
3. Resolver para la frecuencia $\omega_{cr}$ usando la fila auxiliar.
4. $T_{cr} = 2\pi / \omega_{cr}$.

**Ejemplo:** $G(s) = \dfrac{1}{s(s+1)(s+4)}$.

Ecuacion caracteristica: $s^3 + 5s^2 + 4s + K_p = 0$.

Tabla de Routh:
- $s^3$: 1, 4
- $s^2$: 5, $K_p$
- $s^1$: $(20 - K_p)/5$, 0
- $s^0$: $K_p$

Para estabilidad marginal: $(20 - K_p)/5 = 0 \implies K_{cr} = 20$.

Frecuencia: $5s^2 + 20 = 0 \implies s^2 = -4 \implies \omega_{cr} = 2$ rad/s.

$$\boxed{K_{cr} = 20, \quad T_{cr} = \frac{2\pi}{2} = \pi \approx 3.14 \text{ s}}$$

### Tipo 9: Efecto de perturbacion con y sin accion integral

**Metodologia:**
1. Calcular la FT de perturbacion: $Y(s)/D(s) = G(s)/(1 + C(s)G(s))$.
2. Con P: error permanente ante perturbacion escalon $\neq 0$.
3. Con PI: la accion integral rechaza la perturbacion ($e_{ss} = 0$).

**Ejemplo:** $G(s) = \dfrac{1}{s+1}$, perturbacion escalon $D(s) = 1/s$.

- Con P ($K_p = 5$): $y_d(\infty) = \dfrac{G(0)}{1 + K_p G(0)} = \dfrac{1}{1+5} = \dfrac{1}{6}$.
- Con PI: la accion integral fuerza $y_d(\infty) = 0$.

$$\boxed{\text{PI rechaza perturbaciones constantes: } e_{ss,d} = 0}$$

### Tipo 10: Calcular funcion de transferencia en lazo cerrado con PID

**Metodologia:**
1. Expresar $C(s)$ como cociente de polinomios.
2. Calcular $T(s) = \dfrac{C(s)G(s)}{1 + C(s)G(s)}$.
3. Simplificar.

**Ejemplo:** $G(s) = \dfrac{1}{s+2}$, $C(s) = 3\left(1 + \dfrac{1}{2s} + 0.5s\right) = \dfrac{3(s^2 + 2s + 1)}{2s} = \dfrac{3(s+1)^2}{2s}$.

$$T(s) = \frac{\dfrac{3(s+1)^2}{2s(s+2)}}{1 + \dfrac{3(s+1)^2}{2s(s+2)}} = \frac{3(s+1)^2}{2s(s+2) + 3(s+1)^2} = \frac{3(s+1)^2}{2s^2 + 4s + 3s^2 + 6s + 3}$$

$$\boxed{T(s) = \frac{3s^2 + 6s + 3}{5s^2 + 10s + 3}}$$

### Tipo 11: Estabilidad del sistema con controlador PID

**Metodologia:**
1. Obtener la ecuacion caracteristica del lazo cerrado.
2. Aplicar criterio de Routh-Hurwitz.
3. Determinar rango de parametros para estabilidad.

**Ejemplo:** $G(s) = \dfrac{1}{s(s+3)}$, $C(s) = K_p(1 + 0.5s) = K_p(0.5s + 1)$.

Ecuacion caracteristica: $s^2 + 3s + K_p \cdot 0.5s + K_p = s^2 + (3 + 0.5K_p)s + K_p = 0$.

Routh: todos los coeficientes positivos $\implies K_p > 0$.

$$\boxed{\text{Estable para todo } K_p > 0}$$

### Tipo 12: Comparar controladores y elegir el mejor

**Metodologia:**
1. Simular la respuesta al escalon con cada controlador.
2. Medir: tiempo de subida $t_r$, sobreoscilacion $M_p$, tiempo de establecimiento $t_s$, error $e_{ss}$.
3. Comparar con las especificaciones y elegir.

**Ejemplo:** $G(s) = \dfrac{2}{(s+1)(s+2)}$. Especificaciones: $e_{ss} = 0$, $M_p < 25\%$, $t_s < 5$ s.

- P ($K_p=10$): $e_{ss} = 0.23 \neq 0$ $\Rightarrow$ NO cumple.
- PI ($K_p=10$, $T_I=1.5$): $e_{ss} = 0$, $M_p$ puede ser alto.
- PID ($K_p=10$, $T_I=1.5$, $T_D=0.2$): $e_{ss} = 0$, $M_p$ controlado.

$$\boxed{\text{El PID cumple todas las especificaciones}}$$

In [ ]:
# Verificacion Tipo 12
G_t12_num = [2.0]
G_t12_den = [1.0, 3.0, 2.0]
t = np.linspace(0, 10, 3000)

configs = {
    'P':   ([10], [1.0]),
    'PI':  ([10*1.5, 10], [1.5, 0.0]),
    'PID': ([10*1.5*0.2, 10*1.5, 10], [1.5, 0.0]),
}
colores_t12 = {'P': '#1f77b4', 'PI': '#d62728', 'PID': '#2ca02c'}

fig, ax = plt.subplots(figsize=(12, 6))

for nombre, (c_n, c_d) in configs.items():
    t_o, y_o = step_response_cl(c_n, c_d, G_t12_num, G_t12_den, t=t)
    e_ss = abs(1.0 - y_o[-1])
    Mp_val = (np.max(y_o) - 1.0) / 1.0 * 100 if np.max(y_o) > 1.0 else 0
    # Tiempo de establecimiento (2%)
    idx_ts = np.where(np.abs(y_o - y_o[-1]) > 0.02 * y_o[-1])[0]
    ts_val = t_o[idx_ts[-1]] if len(idx_ts) > 0 else 0
    ax.plot(t_o, y_o, color=colores_t12[nombre], linewidth=2.5,
            label=rf'{nombre}: $e_{{ss}}={e_ss:.3f}$, $M_p={Mp_val:.1f}\%$, $t_s={ts_val:.2f}$ s')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(1.25, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.axvline(5.0, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_xlabel(r'Tiempo $t$ (s)')
ax.set_ylabel(r'$y(t)$')
ax.set_title(r'Tipo 12: Comparacion y seleccion del controlador - $G(s) = 2/[(s+1)(s+2)]$')
ax.legend(fontsize=10)
ax.set_xlim(0, 10)

plt.tight_layout()
plt.show()

---

## 13. Tablas resumen

### Tabla 1: Acciones de control y sus efectos

| Accion | Funcion de transferencia | Efecto sobre velocidad | Efecto sobre $e_{ss}$ | Efecto sobre estabilidad |
|--------|--------------------------|------------------------|----------------------|-------------------------|
| P      | $C(s) = K_p$             | Aumenta               | Reduce (no elimina en tipo 0)  | Puede degradar          |
| I      | $C(s) = K_i/s$           | Reduce                | **Elimina**           | Degrada                 |
| D      | $C(s) = K_D s$           | Poco efecto            | Sin efecto            | **Mejora**              |
| PI     | $K_p(1 + 1/(T_Is))$      | Reduce ligeramente    | **Elimina**           | Degrada ligeramente     |
| PD     | $K_p(1 + T_Ds)$          | Similar a P            | No mejora             | **Mejora**              |
| PID    | $K_p(1 + 1/(T_Is) + T_Ds)$ | Buena               | **Elimina**           | **Mejora**              |

### Tabla 2: Efecto de aumentar cada parametro del PID

| Parametro | Tiempo de subida | Sobreoscilacion | Tiempo de establecimiento | Error $e_{ss}$ |
|-----------|-----------------|-----------------|--------------------------|----------------|
| $\uparrow K_p$ | Disminuye | Aumenta | Pequeno cambio | Disminuye |
| $\uparrow T_I$ (mas I) | Disminuye | Aumenta | Aumenta | Elimina |
| $\uparrow T_D$ (mas D) | Pequeno cambio | Disminuye | Disminuye | Sin efecto |

*Nota:* $\uparrow T_I$ significa $\downarrow$ la constante $T_I$ (mas accion integral), lo cual corresponde a $\uparrow K_i = K_p / T_I$.

### Tabla 3: Ziegler-Nichols - Metodo 1 (Curva de reaccion)

| Controlador | $K_p$ | $T_I$ | $T_D$ |
|-------------|-------|-------|-------|
| P           | $T/(KL)$ | $\infty$ | $0$ |
| PI          | $0.9 \cdot T/(KL)$ | $L/0.3 = 3.33L$ | $0$ |
| PID         | $1.2 \cdot T/(KL)$ | $2L$ | $0.5L$ |

### Tabla 4: Ziegler-Nichols - Metodo 2 (Oscilacion sostenida)

| Controlador | $K_p$ | $T_I$ | $T_D$ |
|-------------|-------|-------|-------|
| P           | $0.5 K_{cr}$ | $\infty$ | $0$ |
| PI          | $0.45 K_{cr}$ | $T_{cr}/1.2$ | $0$ |
| PID         | $0.6 K_{cr}$ | $T_{cr}/2$ | $T_{cr}/8$ |

### Tabla 5: Guia rapida de seleccion de controlador

| Situacion | Controlador recomendado | Razon |
|-----------|------------------------|-------|
| Solo necesita velocidad, tolera error | P | Sencillo |
| Necesita error cero, tolera sobreoscilacion | PI | Elimina error |
| Necesita baja sobreoscilacion, tolera error | PD | Amortigua |
| Necesita error cero Y baja sobreoscilacion | PID | Combina todo |
| Planta con retardo puro | ZN Metodo 1 | Aprovecha parametros de retardo |
| Se puede experimentar con la planta | ZN Metodo 2 | Obtiene Kcr, Tcr experimentalmente |

---

## Resumen final

En este tema hemos estudiado:

1. **Control ON-OFF**: el mas simple, genera oscilacion permanente.
2. **Controlador P**: rapido pero con error permanente (en planta tipo 0).
3. **Controlador I**: elimina error pero es lento y oscilatorio.
4. **Controlador D**: anticipa el error, no se usa solo.
5. **PD**: reduce sobreoscilacion sin mejorar error.
6. **PI**: elimina error a costa de mas sobreoscilacion.
7. **PID**: combina las tres acciones para el mejor compromiso.
8. **Ziegler-Nichols Metodo 1**: sintonizacion a partir de la curva de reaccion ($K$, $L$, $T$).
9. **Ziegler-Nichols Metodo 2**: sintonizacion a partir de la oscilacion sostenida ($K_{cr}$, $T_{cr}$).

Las **formulas clave** del PID:

$$\boxed{C(s) = K_p\left(1 + \frac{1}{T_I s} + T_D s\right)}$$